# DS605 Lab 4: Test the From-Scratch Model on `df2.csv`

`df2.csv` is in the same *raw cleaned* format as before feature engineering (not yet one-hot encoded). This notebook re-applies the exact same feature engineering as training -- using the saved model's `feature_names` list as the source of truth for which categories/bigrams to use -- then evaluates the already-trained model on it.


In [ ]:
import numpy as np
import pandas as pd
import pickle

# Load the trained model artifact (weights + scaling params + expected feature list)
with open('airbnb_price_model_scratch.pkl', 'rb') as f:
    artifact = pickle.load(f)

weights = artifact['weights']
train_mean = artifact['train_mean']
train_std = artifact['train_std']
feature_names = artifact['feature_names']   # exact column order the model was trained on

df2 = pd.read_csv('df2.csv')
df2.head()


## 1. Rebuild the Same Features Used in Training

In [ ]:
df2_features = df2.copy()

# --- Low-cardinality categoricals: room_type, neighbourhood_group ---
df2_features = pd.get_dummies(df2_features, columns=['room_type', 'neighbourhood_group'], drop_first=True, dtype=int)

# --- High-cardinality neighbourhood: bucket using the SAME categories the model saw ---
# Recover which neighbourhoods were kept as their own category from feature_names
# (any column the model expects that starts with "neighbourhood_grouped_").
known_neighbourhoods = [
    c[len('neighbourhood_grouped_'):] for c in feature_names if c.startswith('neighbourhood_grouped_')
]

df2_features['neighbourhood_grouped'] = np.where(
    df2_features['neighbourhood'].isin(known_neighbourhoods),
    df2_features['neighbourhood'],
    'Other'
)
df2_features = pd.get_dummies(df2_features, columns=['neighbourhood_grouped'], drop_first=True, dtype=int)
df2_features = df2_features.drop(columns=['neighbourhood'])

# --- host_id: dropped, same as training (identifier, not a feature) ---
df2_features = df2_features.drop(columns=['host_id'])

# --- Bigram keyword flags: recover the exact top-15 phrases from feature_names ---
bigram_phrases = [
    c[len('name_bigram_'):].replace('_', ' ') for c in feature_names if c.startswith('name_bigram_')
]

desc = df2_features['name'].fillna('').str.lower()
for phrase in bigram_phrases:
    col_name = f"name_bigram_{phrase.replace(' ', '_')}"
    df2_features[col_name] = desc.str.contains(phrase, regex=False).astype(int)

# --- Target: log-transform price the same way as training, for evaluation ---
y_true = np.log1p(df2_features['price'])

df2_features.head()


## 2. Align Columns to Match the Trained Model

Reindex to the model's exact `feature_names` list and order -- any expected column missing here (e.g. a neighbourhood category not present in this dataset) is filled with 0, and any extra column (e.g. `name`, `price`) is dropped.


In [ ]:
X_new = df2_features.reindex(columns=feature_names, fill_value=0).to_numpy(dtype=float)
y_new = y_true.to_numpy(dtype=float)

print("Aligned feature matrix shape:", X_new.shape, " (expected:", len(feature_names), "columns)")


## 3. Standardize (using the TRAINING mean/std, not df2's own) and Predict

In [ ]:
def standardize_apply(X, mean, std):
    return (X - mean) / std

def add_bias(X):
    return np.hstack([np.ones((X.shape[0], 1)), X])

def predict_linear_regression(X, weights):
    return add_bias(X) @ weights


X_new_scaled = standardize_apply(X_new, train_mean, train_std)
y_pred = predict_linear_regression(X_new_scaled, weights)


## 4. Evaluate (metrics from scratch, same as training)

In [ ]:
def root_mean_squared_error_manual(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mean_absolute_error_manual(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def r2_score_manual(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

def expm1_manual(x):
    return np.exp(x) - 1


r2 = r2_score_manual(y_new, y_pred)
rmse_log = root_mean_squared_error_manual(y_new, y_pred)

pred_dollars = expm1_manual(y_pred)
actual_dollars = expm1_manual(y_new)
mae_dollars = mean_absolute_error_manual(actual_dollars, pred_dollars)
rmse_dollars = root_mean_squared_error_manual(actual_dollars, pred_dollars)

print(f"R2 on df2:          {r2:.4f}")
print(f"RMSE (log price):   {rmse_log:.4f}")
print(f"MAE (\$):            {mae_dollars:.2f}")
print(f"RMSE (\$):           {rmse_dollars:.2f}")


**Note:** `neighbourhood` bucketing above can't perfectly recover the one category that got dropped as the encoding baseline during training (a side effect of `drop_first=True`) -- listings in that one specific neighbourhood get bucketed as `"Other"` here instead. This affects at most one neighbourhood's rows and shouldn't meaningfully change the results, but it's worth knowing if you see a small discrepancy.
